In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/santander-product-recommendation/test_ver2.csv.zip
/kaggle/input/competitions/santander-product-recommendation/sample_submission.csv.zip
/kaggle/input/competitions/santander-product-recommendation/train_ver2.csv.zip


In [2]:
#import
import pandas as pd
import numpy as np
import lightgbm as lgb

In [3]:
#train.csv 読み込み
train = pd.read_csv('/kaggle/input/competitions/santander-product-recommendation/train_ver2.csv.zip')
test = pd.read_csv('/kaggle/input/competitions/santander-product-recommendation/test_ver2.csv.zip')

/tmp/ipykernel_16/3920408747.py:2: DtypeWarning: Columns (5,8,11,15) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/kaggle/input/competitions/santander-product-recommendation/train_ver2.csv.zip')
/tmp/ipykernel_16/3920408747.py:3: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv('/kaggle/input/competitions/santander-product-recommendation/test_ver2.csv.zip')


In [4]:
#データ確認
print(train.shape)
display(train.head())
print(test.shape)
display(test.head())

(13647309, 48)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0.0,6,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0


(929615, 24)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,...,indext,conyuemp,canal_entrada,indfall,tipodom,cod_prov,nomprov,ind_actividad_cliente,renta,segmento
0,2016-06-28,15889,F,ES,V,56,1995-01-16,0,256,1,...,N,N,KAT,N,1,28.0,MADRID,1,326124.90,01 - TOP
1,2016-06-28,1170544,N,ES,H,36,2013-08-28,0,34,1,...,N,NaN,KAT,N,1,3.0,ALICANTE,0,NA,02 - PARTICULARES
2,2016-06-28,1170545,N,ES,V,22,2013-08-28,0,34,1,...,N,NaN,KHE,N,1,15.0,"CORUÑA, A",1,NA,03 - UNIVERSITARIO
3,2016-06-28,1170547,N,ES,H,22,2013-08-28,0,34,1,...,N,NaN,KHE,N,1,8.0,BARCELONA,0,148402.98,03 - UNIVERSITARIO
4,2016-06-28,1170548,N,ES,H,22,2013-08-28,0,34,1,...,N,NaN,KHE,N,1,7.0,"BALEARS, ILLES",0,106885.80,03 - UNIVERSITARIO


In [5]:
#メモリ削除の為の関数
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            pass

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

In [6]:
#メモリ削除
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

Memory usage of dataframe is 4997.79 MB


/tmp/ipykernel_16/481835741.py:22: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:


Memory usage after optimization is: 2342.72 MB
Decreased by 53.1%
Memory usage of dataframe is 170.22 MB
Memory usage after optimization is: 122.34 MB
Decreased by 28.1%


In [7]:
# ① 商品カラムの取得（最優先）
product_cols = [c for c in train.columns if c.startswith('ind_') and c.endswith('_ult1')]

# ② 商品カラムの NaN を埋める（LightGBM が y の NaN を許さない）
train[product_cols] = train[product_cols].fillna(0)

# ③ カテゴリ列の前処理
cat_cols = [
    'ind_empleado','pais_residencia','sexo','ind_nuevo','indrel',
    'indrel_1mes','tiprel_1mes','indresi','indext','conyuemp',
    'canal_entrada','indfall','tipodom','cod_prov','nomprov',
    'ind_actividad_cliente','segmento'
]

def preprocess(df):
    for c in cat_cols:
        df[c] = df[c].astype(str)
        df[c] = df[c].astype('category').cat.codes.astype('int32')
    return df

train = preprocess(train)
test  = preprocess(test)

# ④ 数値列の NaN を埋める（product_cols が必要なのでここ）
num_cols = [c for c in train.columns if c not in cat_cols + product_cols]

train[num_cols] = train[num_cols].fillna(-1)
test[num_cols]  = test[num_cols].fillna(-1)

In [8]:
#train の lag_1 を作る（shift）
train = train.sort_values(['ncodpers', 'fecha_dato'])

for c in product_cols:
    train[c + '_lag1'] = train.groupby('ncodpers')[c].shift(1).fillna(0)

In [9]:
##test の lag_1 を作る（train の最後の月から）
# train_last
train_last = train[train['fecha_dato'] == '2016-05-28']

# 商品カラム（train だけを見る）
product_cols = [c for c in train.columns if c.startswith('ind_') and c.endswith('_ult1')]

# merge
test = test.merge(
    train_last[['ncodpers'] + product_cols],
    on='ncodpers',
    how='left',
    suffixes=('', '_lag1')
)

# lag カラムが存在しない場合は作る
for c in product_cols:
    lag_col = c + '_lag1'
    if lag_col not in test.columns:
        test[lag_col] = 0
    test[lag_col] = test[lag_col].fillna(0)

In [10]:
#特徴量リスト（lag_1 のみ）
features = [c + '_lag1' for c in product_cols]

In [11]:
#train/valid 分割
train_part = train[train['fecha_dato'] < '2015-06-28']
valid_part = train[train['fecha_dato'] == '2015-06-28']

In [12]:
#パラメータ設定
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "learning_rate": 0.05,
    "num_leaves": 32,
    "random_state": 123,
    "importance_type": "gain"
}


In [13]:
#LightGBM 学習関数
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

def train_lgb(train_part, valid_part, features, target, params):

    model = LGBMClassifier(
        **params,
        n_estimators=5000  # early stopping で自動調整
    )

    model.fit(
        train_part[features],
        train_part[target],
        eval_set=[(valid_part[features], valid_part[target])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
    )

    valid_pred = model.predict_proba(valid_part[features])[:, 1]
    auc = roc_auc_score(valid_part[target], valid_pred)

    return model, valid_pred, auc


In [14]:
#24モデル学習ループ
models = {}
valid_preds = {}
valid_aucs = {}

for prod in product_cols:  # ← 24商品のリスト
    print(f"\n===== Training model for {prod} =====")

    model, valid_pred, auc = train_lgb(
        train_part=train_part,
        valid_part=valid_part,
        features=features,
        target=prod,
        params=params
    )

    models[prod] = model
    valid_preds[prod] = valid_pred
    valid_aucs[prod] = auc



===== Training model for ind_ahor_fin_ult1 =====
[LightGBM] [Info] Number of positive: 438, number of negative: 3143946
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.307092 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 48
[LightGBM] [Info] Number of data points in the train set: 3144384, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000139 -> initscore=-8.878770
[LightGBM] [Info] Start training from score -8.878770
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.999998	valid_0's binary_logloss: 0.00234028

===== Training model for ind_aval_fin_ult1 =====
[LightGBM] [Info] Number of positive: 102, number of negative: 3144282
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.299390 seconds.
You ca

In [15]:
#AUC の一覧を確認
print("\n=== AUC summary ===")
for prod in product_cols:
    print(f"{prod}: {valid_aucs[prod]:.5f}")


=== AUC summary ===
ind_ahor_fin_ult1: 1.00000
ind_aval_fin_ult1: 0.99994
ind_cco_fin_ult1: 0.98203
ind_cder_fin_ult1: 0.98951
ind_cno_fin_ult1: 0.99271
ind_ctju_fin_ult1: 0.99969
ind_ctma_fin_ult1: 0.98895
ind_ctop_fin_ult1: 0.99932
ind_ctpp_fin_ult1: 0.99918
ind_deco_fin_ult1: 0.95043
ind_deme_fin_ult1: 0.99109
ind_dela_fin_ult1: 0.99196
ind_ecue_fin_ult1: 0.99680
ind_fond_fin_ult1: 0.99636
ind_hip_fin_ult1: 0.99972
ind_plan_fin_ult1: 0.99901
ind_pres_fin_ult1: 0.99917
ind_reca_fin_ult1: 0.98894
ind_tjcr_fin_ult1: 0.98015
ind_valo_fin_ult1: 0.99766
ind_viv_fin_ult1: 0.99977
ind_nomina_ult1: 0.98539
ind_nom_pens_ult1: 0.98516
ind_recibo_ult1: 0.96938


In [16]:
#モデル保存
import pickle

for prod in product_cols:
    with open(f"model_{prod}.pkl", "wb") as f:
        pickle.dump(models[prod], f)

In [17]:
#推論テンプレ（24モデル × test）
# test 用の予測
test_preds = {}

for prod in product_cols:
    print(f"Predicting for {prod} ...")
    model = models[prod]  # すでに学習済みのモデル
    test_preds[prod] = model.predict_proba(test[features])[:, 1]

Predicting for ind_ahor_fin_ult1 ...
Predicting for ind_aval_fin_ult1 ...
Predicting for ind_cco_fin_ult1 ...
Predicting for ind_cder_fin_ult1 ...
Predicting for ind_cno_fin_ult1 ...
Predicting for ind_ctju_fin_ult1 ...
Predicting for ind_ctma_fin_ult1 ...
Predicting for ind_ctop_fin_ult1 ...
Predicting for ind_ctpp_fin_ult1 ...
Predicting for ind_deco_fin_ult1 ...
Predicting for ind_deme_fin_ult1 ...
Predicting for ind_dela_fin_ult1 ...
Predicting for ind_ecue_fin_ult1 ...
Predicting for ind_fond_fin_ult1 ...
Predicting for ind_hip_fin_ult1 ...
Predicting for ind_plan_fin_ult1 ...
Predicting for ind_pres_fin_ult1 ...
Predicting for ind_reca_fin_ult1 ...
Predicting for ind_tjcr_fin_ult1 ...
Predicting for ind_valo_fin_ult1 ...
Predicting for ind_viv_fin_ult1 ...
Predicting for ind_nomina_ult1 ...
Predicting for ind_nom_pens_ult1 ...
Predicting for ind_recibo_ult1 ...


In [18]:
#MAP@7 用のランキング生成テンプレ
import numpy as np
import pandas as pd

# 24商品の確率を行方向に積む
pred_matrix = np.vstack([test_preds[prod] for prod in product_cols]).T

# 各行で確率の高い順に7つ選ぶ
top7_idx = np.argsort(-pred_matrix, axis=1)[:, :7]

# 商品名に変換
top7_products = [
    " ".join([product_cols[i] for i in row])
    for row in top7_idx
]

# submission DataFrame
submission = pd.DataFrame({
    "ncodpers": test["ncodpers"],
    "added_products": top7_products
})

In [19]:
#CSV 出力テンプレ
submission.to_csv("submission.csv", index=False)